# StreamLoop — Tuning the Churn Model

Train/test split first → baseline (defaults) → RandomizedSearchCV → narrowed GridSearchCV  
→ inspect `cv_results_` for **mean vs fold stability** → choose final model → score test once.

**Scoring:** `recall` (catch churners). Searches fit on **`X_train` only**. Test set touched twice only.


In [1]:
import json
from pathlib import Path

from run_churn_tuning import (
    CANDIDATES_PATH,
    DATA_URL,
    REPORT_PATH,
    SCORING,
    RANDOM_PARAM_DISTRIBUTIONS,
    build_pipeline,
    cv_results_table,
    evaluate,
    load_raw,
    minimal_clean,
    narrow_param_grid,
    normalize_params,
    select_final_candidate,
    write_tuning_report,
)
from sklearn.base import clone
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

RANDOM_STATE = 42


## 1–2. Load + minimal clean

In [2]:
raw = load_raw(DATA_URL)
X, y = minimal_clean(raw)
print(raw.shape, "churn_rate=", round(float(y.mean()), 3), "TotalCharges_na=", int(X["TotalCharges"].isna().sum()))


(7043, 21) churn_rate= 0.265 TotalCharges_na= 11


## 3. Train/test split before model work

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(len(X_train), len(X_test))


5634 1409


## 4–5. Baseline (defaults) — first test-set touch

In [4]:
baseline = build_pipeline(X_train)
baseline.fit(X_train, y_train)
baseline_metrics = evaluate(baseline, X_test, y_test)
Path("baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2) + "\n")
baseline_metrics


{'accuracy': 0.7778566359119943,
 'precision': 0.6033898305084746,
 'recall': 0.47593582887700536,
 'f1': 0.5321375186846039,
 'roc_auc': 0.8161590327830738}

## 6a–6c. Search on training split only (`n_jobs=1`)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    build_pipeline(X_train),
    param_distributions=RANDOM_PARAM_DISTRIBUTIONS,
    n_iter=16,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
    refit=True,
)
random_search.fit(X_train, y_train)

narrowed = narrow_param_grid(random_search.best_params_)
grid_search = GridSearchCV(
    build_pipeline(X_train),
    param_grid=narrowed,
    scoring=SCORING,
    cv=cv,
    n_jobs=1,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train, y_train)
print("random CV", round(float(random_search.best_score_), 4))
print("grid mean-best CV", round(float(grid_search.best_score_), 4), grid_search.best_params_)


Fitting 5 folds for each of 16 candidates, totalling 80 fits


Fitting 5 folds for each of 18 candidates, totalling 90 fits


random CV 0.794
grid mean-best CV 0.804 {'model__bootstrap': True, 'model__class_weight': 'balanced', 'model__criterion': 'log_loss', 'model__max_depth': 6, 'model__max_features': 'log2', 'model__min_samples_leaf': 8, 'model__min_samples_split': 10, 'model__n_estimators': 50}


## 6d. Inspect `cv_results_` — don't take face value

Compare top candidates on **mean** CV recall **and fold std**. Choose highest mean, or a
slightly lower but more stable option when the trade-off is clear.


In [6]:
top = cv_results_table(grid_search)
display(top.head(5)[["rank", "mean_test_score", "std_test_score", "mean_minus_std", "fold_scores", "params"]])

final_params, selection_reason = select_final_candidate(top)
print(selection_reason)
print(final_params)

CANDIDATES_PATH.write_text(json.dumps(top.head(8).to_dict(orient="records"), indent=2) + "\n")

grid_best = normalize_params(grid_search.best_params_)
if final_params == grid_best:
    final_model = grid_search.best_estimator_  # already refit on train
    fitted_via = "grid_search.best_estimator_ (refit=True on train)"
else:
    final_model = clone(build_pipeline(X_train)).set_params(**final_params)
    final_model.fit(X_train, y_train)
    fitted_via = "clone+set_params+fit on X_train (stabler ≠ mean-best)"
fitted_via


,rank,mean_test_score,std_test_score,mean_minus_std,fold_scores,params
0,1,0.804013,0.024449,0.779564,"[0.8260869565217391, 0.7658862876254181, 0.806...","{'model__bootstrap': True, 'model__class_weigh..."
1,2,0.804013,0.029049,0.774964,"[0.822742474916388, 0.7591973244147158, 0.8093...","{'model__bootstrap': True, 'model__class_weigh..."
2,3,0.802676,0.032905,0.769770,"[0.8127090301003345, 0.7591973244147158, 0.806...","{'model__bootstrap': True, 'model__class_weigh..."
3,4,0.801338,0.030987,0.770351,"[0.8060200668896321, 0.7625418060200669, 0.795...","{'model__bootstrap': True, 'model__class_weigh..."
4,5,0.800669,0.032536,0.768133,"[0.802675585284281, 0.7658862876254181, 0.7993...","{'model__bootstrap': True, 'model__class_weigh..."


Highest mean CV recall (0.8040 ± 0.0244). No nearby candidate offered a clear stability gain for a small mean trade-off.
{'model__bootstrap': True, 'model__class_weight': 'balanced', 'model__criterion': 'log_loss', 'model__max_depth': 6, 'model__max_features': 'log2', 'model__min_samples_leaf': 8, 'model__min_samples_split': 10, 'model__n_estimators': 50}


'grid_search.best_estimator_ (refit=True on train)'

## 7. Final test evaluation — second and last test-set touch

In [7]:
tuned_final = evaluate(final_model, X_test, y_test)
print("baseline", baseline_metrics)
print("tuned   ", tuned_final)

report = write_tuning_report(
    baseline=baseline_metrics,
    tuned=tuned_final,
    final_params=final_params,
    selection_reason=selection_reason,
    top_candidates=top.head(8).to_dict(orient="records"),
    random_best_cv=float(random_search.best_score_),
    grid_best_cv=float(grid_search.best_score_),
)
print("Wrote", REPORT_PATH)
Path("tuning_metrics.json").write_text(
    json.dumps(
        {
            "scoring": SCORING,
            "search_fit_on": "X_train_only",
            "test_touches": ["baseline", "tuned_final"],
            "baseline": baseline_metrics,
            "selection": {"final_params": final_params, "reason": selection_reason, "fitted_via": fitted_via},
            "tuned_final": tuned_final,
        },
        indent=2,
    )
    + "\n"
)
tuned_final


baseline {'accuracy': 0.7778566359119943, 'precision': 0.6033898305084746, 'recall': 0.47593582887700536, 'f1': 0.5321375186846039, 'roc_auc': 0.8161590327830738}
tuned    {'accuracy': 0.7366926898509581, 'precision': 0.5025210084033613, 'recall': 0.7994652406417112, 'f1': 0.6171310629514963, 'roc_auc': 0.8389250561884833}
Wrote /workspace/data/notebooks/churn-model-tuning/tuning_report.md


{'accuracy': 0.7366926898509581,
 'precision': 0.5025210084033613,
 'recall': 0.7994652406417112,
 'f1': 0.6171310629514963,
 'roc_auc': 0.8389250561884833}